# DOFBOT Live Camera Feed with Tracking and FPS

Position the camera securely so the table workspace is visible, then run the cells in order.

Image resolution used: **640 x 480 pixels**.

In [2]:
import time
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display


def bgr8_to_jpeg(frame, quality=80):
    success, encoded = cv2.imencode(".jpg", frame, [int(cv2.IMWRITE_JPEG_QUALITY), quality])
    if not success:
        raise RuntimeError("Could not encode camera frame as JPEG.")
    return encoded.tobytes()


# HSV ranges for common colored blocks. Adjust these values for your lighting.
COLOR_RANGES = {
    "red": [
        (np.array([0, 100, 70]), np.array([10, 255, 255])),
        (np.array([170, 100, 70]), np.array([179, 255, 255])),
    ],
    "green": [
        (np.array([35, 60, 50]), np.array([85, 255, 255])),
    ],
    "blue": [
        (np.array([90, 70, 50]), np.array([135, 255, 255])),
    ],
    "yellow": [
        (np.array([18, 80, 70]), np.array([38, 255, 255])),
    ],
}

LABEL_COLORS = {
    "red": (0, 0, 255),
    "green": (0, 180, 0),
    "blue": (255, 0, 0),
    "yellow": (0, 220, 220),
}

# Tracking state: IDs increase and are never reused during this notebook run.
tracks = {}
next_track_id = 1
MAX_TRACK_DISTANCE = 100
MAX_MISSED_FRAMES = 10


def detect_and_annotate(frame, minimum_area=1200):
    """Detect colored blocks, keep IDs, and draw category, ID, and pixel center."""
    global next_track_id, tracks

    annotated = frame.copy()
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    detections = []

    for label, ranges in COLOR_RANGES.items():
        mask = np.zeros(hsv.shape[:2], dtype=np.uint8)
        for lower, upper in ranges:
            mask |= cv2.inRange(hsv, lower, upper)

        kernel = np.ones((5, 5), np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for contour in contours:
            if cv2.contourArea(contour) < minimum_area:
                continue

            x, y, width, height = cv2.boundingRect(contour)
            detections.append({
                "label": label,
                "bbox": (x, y, width, height),
                "center": (int(x + width / 2), int(y + height / 2)),
            })

    for track in tracks.values():
        track["missed"] += 1

    available_track_ids = set(tracks)
    for detection in detections:
        label = detection["label"]
        center = np.array(detection["center"])
        candidates = []

        for track_id in available_track_ids:
            track = tracks[track_id]
            if track["label"] != label:
                continue
            distance = np.linalg.norm(center - np.array(track["center"]))
            if distance <= MAX_TRACK_DISTANCE:
                candidates.append((distance, track_id))

        if candidates:
            _, track_id = min(candidates)
            track = tracks[track_id]
            track["center"] = detection["center"]
            track["bbox"] = detection["bbox"]
            track["missed"] = 0
            available_track_ids.remove(track_id)
        else:
            track_id = next_track_id
            next_track_id += 1
            tracks[track_id] = {
                "label": label,
                "center": detection["center"],
                "bbox": detection["bbox"],
                "missed": 0,
            }
            available_track_ids.discard(track_id)

        detection["track_id"] = track_id

    tracks = {
        track_id: track
        for track_id, track in tracks.items()
        if track["missed"] <= MAX_MISSED_FRAMES
    }

    for detection in detections:
        x, y, width, height = detection["bbox"]
        label = detection["label"]
        track_id = detection["track_id"]
        center_x, center_y = detection["center"]
        color = LABEL_COLORS[label]
        cv2.rectangle(annotated, (x, y), (x + width, y + height), color, 2)
        cv2.putText(
            annotated,
            f"{label} | ID {track_id} | centre ({center_x}, {center_y}) px",
            (x, max(25, y - 8)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            color,
            2,
            cv2.LINE_AA,
        )

    return annotated


image_widget = widgets.Image(format="jpeg", width=640, height=480)
display(image_widget)

Image(value=b'', format='jpeg', height='480', width='640')

In [3]:
camera = cv2.VideoCapture(0)
camera.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter.fourcc("M", "J", "P", "G"))
camera.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
camera.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
camera.set(cv2.CAP_PROP_FPS, 30)

if not camera.isOpened():
    raise RuntimeError("Could not open the DOFBOT camera. Check the connection and camera index.")

print("Camera opened successfully.")

Camera opened successfully.


In [ ]:
# Run this cell to start the continuous feed with block detection.
# Place at least three colored blocks in view, then use Interrupt Kernel to stop.
try:
    while True:
        ret, frame = camera.read()
        if not ret or frame is None:
            print("No frame received; retrying...")
            time.sleep(0.1)
            continue

        annotated_frame = detect_and_annotate(frame)
        image_widget.value = bgr8_to_jpeg(annotated_frame)
        time.sleep(0.03)
except KeyboardInterrupt:
    print("Live capture stopped.")

In [2]:
# Run this cleanup cell after stopping the live-feed cell.
camera.release()
image_widget.value = b""
print("Camera released.")

NameError: name 'camera' is not defined